# Textvorverarbeitung (Preprocessing)

In diesem Notebook werden die Beschwerdetexte bereinigt und für die spätere NLP-Analyse vorbereitet. Ziel ist es, aus den unstrukturierten Rohtexten saubere und analysierbare Texte zu erzeugen.

In [43]:
import pandas as pd
import re
import nltk
import spacy

In [44]:
df = pd.read_csv("../processed_data/01_loaded_data.csv")

## Vorbereitung der NLP-Bibliotheken

Für die Textverarbeitung werden Stopwörter und Tokenizer aus der Bibliothek NLTK verwendet.

In [45]:
import nltk

nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /Users/manu/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /Users/manu/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

## Laden der Stopwörter

Stopwörter sind häufig vorkommende Wörter wie „the“, „and“ oder „is“, die meist keinen inhaltlichen Mehrwert für die Analyse besitzen und daher entfernt werden.

In [46]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

## Bereinigung der Texte

Die Beschwerdetexte werden vereinheitlicht, indem Großbuchstaben umgewandelt sowie Sonderzeichen und Zahlen entfernt werden.

In [47]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

## Laden des vorbereiteten Datensatzes

Der in Notebook 1 bereinigte Datensatz wird erneut geladen. Dadurch müssen die Rohdaten nicht erneut verarbeitet werden und die weiteren NLP-Schritte können effizient auf den bereits vorbereiteten Daten durchgeführt werden.

In [48]:
df['clean_text'] = df['consumer_complaint_narrative'].apply(clean_text)

## Vergleich von Originaltext und bereinigtem Text

Zur Kontrolle werden die ursprünglichen Beschwerdetexte mit den bereinigten Versionen verglichen.

In [49]:
df[['consumer_complaint_narrative', 'clean_text']].head()

,consumer_complaint_narrative,clean_text
0,These are not my accounts.,these are not my accounts
1,Kindly address this issue on my credit report....,kindly address this issue on my credit report ...
2,"I wrote three requests, the unverified account...",i wrote three requests the unverified accounts...
3,XXXX XXXX has a old account settled in XXXX th...,xxxx xxxx has a old account settled in xxxx th...
4,They call at all hours and on the weekends usi...,they call at all hours and on the weekends usi...


## Laden des spaCy-Sprachmodells

Das spaCy-Modell wird verwendet, um Wörter auf ihre Grundform (Lemma) zu reduzieren.

In [50]:
nlp = spacy.load("en_core_web_sm")

## Lemmatization der Texte

Die Wörter werden auf ihre Grundform reduziert. Zusätzlich werden Stopwörter entfernt, um die Qualität der späteren Analyse zu verbessern.

In [51]:
def lemmatize_text(text):
    doc = nlp(text)

    tokens = [
        token.lemma_
        for token in doc
        if token.text not in stop_words and token.is_alpha
    ]

    return " ".join(tokens)

In [52]:
df['processed_text'] = df['clean_text'].apply(lemmatize_text)

## Kontrolle der vorverarbeiteten Texte

Die bereinigten und lemmatisierten Texte bilden die Grundlage für die spätere Vektorisierung und Themenanalyse.

In [53]:
df[['clean_text', 'processed_text']].head()

,clean_text,processed_text
0,these are not my accounts,account
1,kindly address this issue on my credit report ...,kindly address issue credit report assert acco...
2,i wrote three requests the unverified accounts...,write three request unverified account list st...
3,xxxx xxxx has a old account settled in xxxx th...,xxxx xxxx old account settle xxxx keep reappea...
4,they call at all hours and on the weekends usi...,call hour weekend use various number


## Zusammenfassung der Textvorverarbeitung

Die Beschwerdetexte wurden erfolgreich bereinigt und standardisiert. Sonderzeichen, Zahlen und Stopwörter wurden entfernt sowie Wörter auf ihre Grundform reduziert. Dadurch liegen nun saubere Texte vor, die für die weitere NLP-Analyse verwendet werden können.

## Speicherung der vorverarbeiteten Texte

Die bereinigten und lemmatisierten Texte werden gespeichert, damit sie in den folgenden Schritten direkt für die Vektorisierung und Themenanalyse verwendet werden können.

In [54]:
df.to_csv("../processed_data/02_preprocessed_data.csv", index=False)